# ChatGeo · ต้นแบบตรวจน้ำท่วมจากดาวเทียม Sentinel-1 (ลุ่มน้ำชีตอนล่าง)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Choonahakarn/chatgeo/blob/main/notebooks/flood_s1_chi.ipynb)

**ทำอะไร:** หาภาพเรดาร์ Sentinel-1 ล่าสุดเหนือ ร้อยเอ็ด ยโสธร มหาสารคาม แล้วหาว่าตรงไหนน้ำท่วม (ไม่นับแม่น้ำและหนองบึงที่มีน้ำอยู่แล้ว) ได้ออกมาเป็นไฟล์ GeoJSON ภาพตัวอย่าง แผนที่แบบซูมได้ และพื้นที่น้ำท่วมรายจังหวัด

**ทำไมใช้เรดาร์:** ดาวเทียมเรดาร์เหมือนส่องไฟฉายลงมาเองแล้วรอรับแสงสะท้อน จึงถ่ายได้ทั้งกลางคืนและทะลุเมฆช่วงหน้าฝน ผิวน้ำเรียบเหมือนกระจก สะท้อนคลื่นออกไปทางอื่น ภาพตรงที่เป็นน้ำจึงมืด เราใช้ "ความมืด" นี้หาน้ำ

**วิธีใช้:** เมนู Runtime › Run all แล้วรอราว 5–10 นาที ไม่ต้องมีบัญชีหรือ API key (ใช้ข้อมูลเปิดจาก Microsoft Planetary Computer)

ขั้นตอน: 1) หาภาพล่าสุด → 2) โหลดเรดาร์ + ข้อมูลแหล่งน้ำถาวร + ความสูงพื้นที่ → 3) แยกน้ำท่วม → 4) แปลงเป็นรูปหลายเหลี่ยมและสรุปรายจังหวัด → 5) ดูบนแผนที่เทียบกับสถานีวัดน้ำ ThaiWater

In [ ]:
# ติดตั้งไลบรารี (ครั้งแรกราว 1 นาที)
!pip -q install pystac-client planetary-computer odc-stac rasterio scikit-image folium shapely pyproj

## ตั้งค่า

- `AOI` กรอบพื้นที่ (ลองจิจูดซ้าย, ละติจูดล่าง, ลองจิจูดขวา, ละติจูดบน) ค่าเริ่มต้นคือลุ่มน้ำชีตอนล่าง
- `RES` ความละเอียดเป็นเมตร 30 เร็วและพอดีกับพื้นที่ขนาดนี้ ถ้าพื้นที่เล็กลงใช้ 10–20 ได้
- `THRESHOLD` เส้นแบ่งน้ำ (เดซิเบล) ปล่อย `None` ให้คำนวณเอง หรือใส่เลข เช่น `-19` ถ้าผลดูมากหรือน้อยเกินไป

In [ ]:
AOI = (102.9, 15.5, 104.6, 16.5)   # ลุ่มน้ำชีตอนล่าง: ร้อยเอ็ด ยโสธร มหาสารคาม กาฬสินธุ์บางส่วน
DAYS_BACK = 24                     # หาภาพย้อนหลังกี่วัน
RES = 30                           # เมตรต่อพิกเซล
THRESHOLD = None                   # None = คำนวณเอง (Otsu)
CRS = 'EPSG:32648'                 # UTM 48N เหมาะกับภาคอีสาน
OUT = 'flood_s1_chi'

## ฟังก์ชันหลัก (รันเฉยๆ ไม่ต้องแก้)

In [ ]:
# ฟังก์ชันหลัก (ไม่ต้องแก้)
import json
import math
import re
import urllib.request
from datetime import datetime, timedelta, timezone

import numpy as np
from scipy import ndimage


def _union_bounds_fraction(items, aoi):
    """สัดส่วนพื้นที่ AOI ที่ภาพชุดนี้ครอบคลุม (0–1)"""
    from shapely.geometry import box, shape
    from shapely.ops import unary_union
    a = box(*aoi)
    u = unary_union([shape(it.geometry) for it in items])
    return u.intersection(a).area / a.area


def find_latest_scene(client, aoi, days_back=24, min_cover=0.6):
    """หาวันล่าสุดที่ Sentinel-1 ถ่ายครอบคลุมพื้นที่มากพอ คืนค่า (วันที่, รายการภาพ, สัดส่วนครอบคลุม)"""
    end = datetime.now(timezone.utc)
    start = end - timedelta(days=days_back)
    search = client.search(collections=['sentinel-1-rtc'], bbox=aoi,
                           datetime=start.strftime('%Y-%m-%d') + '/' + end.strftime('%Y-%m-%d'))
    items = list(search.items())
    if not items:
        raise RuntimeError('ไม่พบภาพ Sentinel-1 ในช่วง %d วันที่ผ่านมา ลองเพิ่ม DAYS_BACK' % days_back)
    by_day = {}
    for it in items:
        by_day.setdefault(it.datetime.strftime('%Y-%m-%d'), []).append(it)
    scored = sorted(((d, its, _union_bounds_fraction(its, aoi)) for d, its in by_day.items()), key=lambda x: x[0], reverse=True)
    for d, its, cov in scored:
        if cov >= min_cover:
            return d, its, cov
    return max(scored, key=lambda x: x[2])


def to_db(linear):
    """แปลงค่าการสะท้อนเรดาร์ (linear) เป็นเดซิเบล ค่าที่ใช้ไม่ได้เป็น NaN"""
    lin = np.asarray(linear, dtype='float32')
    out = np.full(lin.shape, np.nan, dtype='float32')
    ok = np.isfinite(lin) & (lin > 0)
    out[ok] = 10.0 * np.log10(lin[ok])
    return out


def speckle_filter(db, size=3):
    """ลดจุดรบกวนแบบเม็ดทราย (speckle) ด้วย median filter โดยไม่ทำให้ NaN ลาม"""
    filled = np.where(np.isfinite(db), db, 0.0)
    med = ndimage.median_filter(filled, size=size)
    return np.where(np.isfinite(db), med, np.nan).astype('float32')


def otsu_threshold(db, lo=-30.0, hi=0.0, clamp=(-24.0, -15.0), fallback=-19.0):
    """หาเส้นแบ่ง 'น้ำ' กับ 'ไม่ใช่น้ำ' จากฮิสโตแกรม (Otsu) แล้วจำกัดให้อยู่ในช่วงที่สมเหตุสมผล"""
    v = db[np.isfinite(db)]
    v = v[(v > lo) & (v < hi)]
    if v.size < 1000:
        return fallback
    try:
        from skimage.filters import threshold_otsu
        t = float(threshold_otsu(v))
    except Exception:  # noqa: BLE001
        return fallback
    return float(min(max(t, clamp[0]), clamp[1]))


def slope_degrees(dem, res):
    """ความชันของพื้นที่ (องศา) จาก DEM ที่อยู่ในพิกัดเมตร"""
    z = np.where(np.isfinite(dem), dem, np.nanmean(dem) if np.isfinite(dem).any() else 0)
    dy, dx = np.gradient(z, res)
    return np.degrees(np.arctan(np.hypot(dx, dy))).astype('float32')


def detect_flood(db, threshold, occurrence=None, slope=None, perm_occ=75, max_slope=5.0, min_pixels=10):
    """น้ำท่วม = ตอนนี้เป็นน้ำ และปกติไม่ใช่แหล่งน้ำถาวร และไม่ใช่พื้นที่ลาดชัน (เงาภูเขาดูเหมือนน้ำ)"""
    valid = np.isfinite(db)
    water = valid & (db < threshold)
    water = ndimage.binary_opening(water, structure=np.ones((3, 3)))
    permanent = np.zeros_like(water) if occurrence is None else (np.nan_to_num(occurrence, nan=0) >= perm_occ)
    steep = np.zeros_like(water) if slope is None else (np.nan_to_num(slope, nan=0) > max_slope)
    flood = water & ~permanent & ~steep
    lab, n = ndimage.label(flood)
    if n:
        sizes = ndimage.sum(flood, lab, index=np.arange(1, n + 1))
        keep = np.zeros(n + 1, dtype=bool)
        keep[1:] = sizes >= min_pixels
        flood = keep[lab]
    return flood, water, permanent, steep


def vectorize(mask, transform, crs, res, date, min_area_km2=0.05, max_features=3000, simplify_m=None):
    """แปลงพื้นที่น้ำท่วม (ภาพ) เป็นรูปหลายเหลี่ยม GeoJSON (ละติจูด/ลองจิจูด)"""
    from rasterio.features import shapes
    from shapely.geometry import shape, mapping
    from shapely.ops import transform as shp_transform
    from pyproj import Transformer
    tr = Transformer.from_crs(crs, 'EPSG:4326', always_xy=True).transform
    polys = []
    for geom, val in shapes(mask.astype('uint8'), mask=mask, transform=transform):
        if not val:
            continue
        g = shape(geom)
        a = g.area / 1e6
        if a < min_area_km2:
            continue
        polys.append((a, g))
    polys.sort(key=lambda x: -x[0])
    feats = []
    for a, g in polys[:max_features]:
        g = g.simplify(simplify_m or res, preserve_topology=True)
        g4326 = shp_transform(tr, g)
        feats.append({'type': 'Feature', 'properties': {'area_km2': round(a, 3), 'date': date},
                      'geometry': json.loads(json.dumps(mapping(g4326), default=float))})
    return {'type': 'FeatureCollection', 'features': feats,
            'properties': {'date': date, 'source': 'Sentinel-1 RTC (Microsoft Planetary Computer)',
                           'method': 'ChatGeo prototype: Otsu threshold on VV dB, minus JRC permanent water and slope > 5°'}}


def round_coords(gj, nd=5):
    def r(c):
        return [r(x) for x in c] if isinstance(c[0], (list, tuple)) else [round(c[0], nd), round(c[1], nd)]
    for f in gj['features']:
        f['geometry']['coordinates'] = r(f['geometry']['coordinates'])
    return gj


def load_cg_th(url='https://raw.githubusercontent.com/Choonahakarn/chatgeo/main/data/th-regions.js', text=None):
    """อ่านขอบเขตภาค/จังหวัดจากไฟล์ th-regions.js ของ ChatGeo"""
    if text is None:
        with urllib.request.urlopen(url, timeout=60) as r:
            text = r.read().decode('utf-8')
    m = re.search(r'window\.CG_TH\s*=\s*(\{.*\})\s*;?\s*$', text, re.S)
    return json.loads(m.group(1)) if m else None


def province_stats(flood_gj, th):
    """พื้นที่น้ำท่วม (ตร.กม.) แยกตามจังหวัด เรียงจากมากไปน้อย"""
    from shapely.geometry import shape
    from shapely.strtree import STRtree
    if not th or 'provinces' not in th:
        return []
    provs = [(f['properties']['name'], shape(f['geometry'])) for f in th['provinces']['features']]
    tree = STRtree([g for _, g in provs])
    tot = {}
    for f in flood_gj['features']:
        g = shape(f['geometry'])
        c = g.representative_point()
        for i in tree.query(c):
            name, pg = provs[int(i)]
            if pg.contains(c):
                tot[name] = tot.get(name, 0) + f['properties']['area_km2']
                break
    return sorted(tot.items(), key=lambda x: -x[1])

## 1) หาภาพเรดาร์ล่าสุดที่ครอบคลุมพื้นที่

In [ ]:
import pystac_client, planetary_computer
client = pystac_client.Client.open('https://planetarycomputer.microsoft.com/api/stac/v1',
                                   modifier=planetary_computer.sign_inplace)
DATE, items, cover = find_latest_scene(client, AOI, DAYS_BACK)
print('ภาพวันที่', DATE, '· จำนวนแผ่นภาพ', len(items), '· ครอบคลุมพื้นที่ %.0f%%' % (cover * 100))
print('วงโคจร:', sorted({it.properties.get('sat:orbit_state', '?') for it in items}))

## 2) โหลดเรดาร์ (VV) แหล่งน้ำถาวร (JRC) และความสูงพื้นที่ (Copernicus DEM)

- **VV** คือความแรงของคลื่นที่สะท้อนกลับ ยิ่งมืดยิ่งน่าจะเป็นน้ำ
- **แหล่งน้ำถาวร** จากสถิติ 30 ปีของ JRC ใช้ตัดแม่น้ำและหนองบึงที่มีน้ำเป็นปกติ
- **ความชัน** ใช้ตัดเงาภูเขา ซึ่งในภาพเรดาร์ก็มืดเหมือนน้ำ

In [ ]:
import numpy as np, odc.stac
ds = odc.stac.load(items, bands=['vv'], crs=CRS, resolution=RES, bbox=AOI, groupby='solar_day')
vv = ds['vv'].isel(time=0).values
gbox = ds.odc.geobox
transform = gbox.affine
db = speckle_filter(to_db(vv))
print('ขนาดภาพ', db.shape, '· พิกเซลที่ใช้ได้ %.0f%%' % (100 * np.isfinite(db).mean()))

def load_like(collection, band, valid):
    try:
        its = list(client.search(collections=[collection], bbox=AOI).items())
        if not its:
            return None
        da = odc.stac.load(its, bands=[band], like=ds)[band].astype('float32')
        da = da.where(valid(da)).max('time')
        return da.values
    except Exception as e:  # ถ้าโหลดไม่ได้ ข้ามขั้นนี้ไปก่อน
        print('โหลด', collection, 'ไม่ได้:', e)
        return None

occ = load_like('jrc-gsw', 'occurrence', lambda d: d <= 100)
dem = load_like('cop-dem-glo-30', 'data', lambda d: d > -1000)
slope = slope_degrees(dem, RES) if dem is not None else None
print('แหล่งน้ำถาวร:', 'มี' if occ is not None else 'ไม่มี', '· ความชัน:', 'มี' if slope is not None else 'ไม่มี')

## 3) แยกพื้นที่น้ำท่วม

น้ำท่วม = ตอนนี้เป็นน้ำ **และ** ปกติไม่ใช่แหล่งน้ำถาวร **และ** ไม่ใช่ที่ลาดชัน แล้วตัดจุดเล็กๆ ที่เป็นสัญญาณรบกวนออก

In [ ]:
thr = THRESHOLD if THRESHOLD is not None else otsu_threshold(db)
flood, water, perm, steep = detect_flood(db, thr, occ, slope)
px_km2 = RES * RES / 1e6
print('เส้นแบ่งน้ำ %.1f dB' % thr)
print('น้ำทั้งหมด %.1f ตร.กม. · น้ำถาวร %.1f ตร.กม. · น้ำท่วม %.1f ตร.กม.' % (water.sum() * px_km2, (water & perm).sum() * px_km2, flood.sum() * px_km2))

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(16, 7))
ax[0].imshow(db, cmap='gray', vmin=-25, vmax=0); ax[0].set_title('Sentinel-1 VV (dB) ' + DATE)
ax[1].imshow(db, cmap='gray', vmin=-25, vmax=0)
ov = np.zeros(db.shape + (4,)); ov[perm & water] = (0.2, 0.5, 1, 0.8); ov[flood] = (1, 0.25, 0.2, 0.85)
ax[1].imshow(ov); ax[1].set_title('red = flood, blue = permanent water')
for a in ax: a.axis('off')
plt.tight_layout(); plt.savefig(OUT + '_' + DATE + '.png', dpi=110); plt.show()

## 4) แปลงเป็นรูปหลายเหลี่ยม (GeoJSON) และสรุปรายจังหวัด

ขอบเขตจังหวัดดึงจากไฟล์ `data/th-regions.js` ของ ChatGeo บน GitHub

In [ ]:
gj = round_coords(vectorize(flood, transform, CRS, RES, DATE))
with open(OUT + '_' + DATE + '.geojson', 'w') as f:
    json.dump(gj, f, ensure_ascii=False)
print('จำนวนปื้นน้ำท่วม', len(gj['features']))
try:
    th = load_cg_th()
    for name, km2 in province_stats(gj, th)[:10]:
        print('  จ.%s  %.1f ตร.กม.' % (name, km2))
except Exception as e:
    print('สรุปรายจังหวัดไม่ได้:', e)

## 5) ดูบนแผนที่ เทียบกับสถานีวัดน้ำ ThaiWater

สามเหลี่ยมสีคือสถานีวัดน้ำจากข้อมูลที่งานเช้าของ ChatGeo เก็บไว้ ถ้าสถานีแถวไหนล้นตลิ่ง (แดง) แต่ดาวเทียมไม่เห็นน้ำท่วม หรือกลับกัน คือจุดที่ควรตรวจดูให้ละเอียด

In [ ]:
import folium, urllib.request
cx, cy = (AOI[0] + AOI[2]) / 2, (AOI[1] + AOI[3]) / 2
m = folium.Map(location=[cy, cx], zoom_start=9, tiles='OpenStreetMap')
folium.GeoJson(gj, name='น้ำท่วมจาก Sentinel-1 ' + DATE,
               style_function=lambda f: {'color': '#E5484D', 'weight': 1, 'fillColor': '#E5484D', 'fillOpacity': 0.45},
               tooltip=folium.GeoJsonTooltip(fields=['area_km2'], aliases=['พื้นที่ (ตร.กม.)'])).add_to(m)
COL = {1: '#B5652B', 2: '#E0A526', 3: '#2FA36B', 4: '#2F7FE0', 5: '#E5484D'}
NAME = {1: 'น้อยวิกฤต', 2: 'น้อย', 3: 'ปกติ', 4: 'น้ำมาก', 5: 'ล้นตลิ่ง'}
try:
    live = json.load(urllib.request.urlopen('https://raw.githubusercontent.com/Choonahakarn/chatgeo/main/data/live.json', timeout=60))
    fg = folium.FeatureGroup(name='สถานีวัดน้ำ ThaiWater')
    for o in live.get('water', []):
        if AOI[0] <= o['lon'] <= AOI[2] and AOI[1] <= o['lat'] <= AOI[3]:
            folium.CircleMarker([o['lat'], o['lon']], radius=6, color='#fff', weight=1, fill=True, fill_opacity=0.95,
                                fill_color=COL.get(o['lv'], '#888'),
                                tooltip='%s (จ.%s) · %s · %s' % (o['name'], o['prov'], NAME.get(o['lv'], '?'), o['time'])).add_to(fg)
    fg.add_to(m)
except Exception as e:
    print('โหลดสถานี ThaiWater ไม่ได้:', e)
folium.Rectangle([[AOI[1], AOI[0]], [AOI[3], AOI[2]]], color='#555', weight=1, fill=False).add_to(m)
folium.LayerControl().add_to(m)
m

## ดาวน์โหลดผลลัพธ์

ได้ 2 ไฟล์: `flood_s1_chi_<วันที่>.geojson` (เอาไปใส่ในแผนที่ ChatGeo ได้) และ `flood_s1_chi_<วันที่>.png` (ภาพตัวอย่าง)

In [ ]:
try:
    from google.colab import files
    files.download(OUT + '_' + DATE + '.geojson')
    files.download(OUT + '_' + DATE + '.png')
except Exception:
    print('ไม่ได้รันใน Colab ไฟล์อยู่ในโฟลเดอร์เดียวกับโน้ตบุ๊กนี้')

## ข้อจำกัดของต้นแบบนี้ (สำคัญ)

- **เป็นต้นแบบเพื่อทดลอง ไม่ใช่ข้อมูลเตือนภัยทางการ** ให้เทียบกับ GISTDA และ ThaiWater เสมอ
- **ในเมืองมักมองไม่เห็นน้ำท่วม** เพราะคลื่นสะท้อนระหว่างตึกกับผิวน้ำแล้วกลับมาสว่าง
- **อาจนับผิดเป็นน้ำ** ในที่เรียบมาก เช่น ลานคอนกรีต ทราย และแปลงนาที่เพิ่งไถหรือเพิ่งปล่อยน้ำเข้า เพราะในภาพมืดเหมือนน้ำ
- **ภาพ Sentinel-1 มาทุกไม่กี่วัน** น้ำท่วมที่เกิดและลดระหว่างรอบถ่ายอาจไม่ถูกจับ

### ต่อยอดได้
- เทียบกับภาพช่วงหน้าแล้งของพื้นที่เดียวกัน แทนการใช้แหล่งน้ำถาวร JRC อย่างเดียว
- ฝึกโมเดล segmentation (เช่น U-Net) ด้วยชุดข้อมูลน้ำท่วมสาธารณะอย่าง Sen1Floods11 แทนการใช้เส้นแบ่งค่าเดียว
- ใช้รายงานจากผู้ใช้และสถานีวัดน้ำเป็นข้อมูลยืนยันผล แล้ววัดความแม่นยำ